# 04.1 — Exploration du retrieval pour le jeu de test RAGAS

**Objectif** : Pour chaque question du jeu de test (notebook 06), afficher les documents parents retournés par FAISS. Permet de vérifier que le bon document est bien retrouvé et d'identifier les cas où le retriever échoue.

In [13]:
import json

from langchain_community.vectorstores import FAISS
import pandas as pd

from notebooks.config import FAISS_INDEX_PATH, MISTRAL_API_KEY, MISTRAL_EMBED_MODEL
from notebooks.mistral import MistralEmbeddings

pd.set_option("display.max_colwidth", 120, "display.max_rows", 20)

embedder = MistralEmbeddings(api_key=MISTRAL_API_KEY, model=MISTRAL_EMBED_MODEL)
vectorstore = FAISS.load_local(str(FAISS_INDEX_PATH), embedder, allow_dangerous_deserialization=True)

parent_docs: dict[str, str] = json.loads(
    (FAISS_INDEX_PATH / "parent_docs.json").read_text(encoding="utf-8")
)

print(f"Index : {vectorstore.index.ntotal} vecteurs | Parents : {len(parent_docs)}")

Index : 350 vecteurs | Parents : 200


In [14]:
K = 10


def explore_query(query: str):
    results = vectorstore.similarity_search(query, k=K)

    print(f"\nREQUÊTE : « {query} »\n")

    seen_uids: set[str] = set()
    rows = []
    for doc in results:
        uid = doc.metadata.get("uid")
        if uid and uid not in seen_uids:
            seen_uids.add(uid)
            parent = parent_docs.get(uid, doc.page_content)
            rows.append({
                "uid": uid,
                "titre": parent.split("\n")[0],
                "extrait": parent[parent.find("\n") + 1:],
            })

    display(pd.DataFrame(rows))

## 1. Exploration par question de test

Pour chaque question, on affiche les `K` documents parents les plus proches. Le tableau montre : uid, titre (première ligne du doc) et un extrait du texte.

In [15]:
explore_query("Où et à quelles dates se déroule la visite-atelier 'Prendre la mesure du temps' ?")


REQUÊTE : « Où et à quelles dates se déroule la visite-atelier 'Prendre la mesure du temps' ? »



,uid,titre,extrait
0,6670838,# Visite-atelier : Prendre la mesure du temps,"\n**Date** : Du 20 septembre 2025 à 12:30 au 21 septembre 2025 à 12:30\n**Lieu** : Musée de la poste, Paris\n**Adres..."
1,43521113,# Visite libre du Mur des je t'aime en présence des artistes,"\n**Date** : Du 19 septembre 2025 à 06:00 au 21 septembre 2025 à 07:00\n**Lieu** : Le mur des je t'aime, Paris\n**Ad..."
2,65315210,# Visite du site historique du Collège de France,"\n**Date** : Du 20 septembre 2025 à 08:00 au 21 septembre 2025 à 08:00\n**Lieu** : Collège de France, Paris\n**Adres..."
3,40948393,# Exposition des travaux des élèves des lycées professionnels et des écoles d’art de l’académie de Paris,"\n**Date** : Du 20 septembre 2025 à 08:00 au 21 septembre 2025 à 08:00\n**Lieu** : La Sorbonne, Paris\n**Adresse** :..."
4,11372405,# Une bibliothèque d'art dans un monument historique,\n**Date** : Du 19 septembre 2026 à 09:00 au 20 septembre 2026 à 09:00\n**Lieu** : Bibliothèque Forney – Hôtel de Se...
5,51382688,# Hercule(s) (7-12 ans en famille),"\n**Date** : Du 8 août 2025 à 12:30 au 13 août 2025 à 12:30\n**Lieu** : Musée des arts et métiers, Paris\n**Adresse*..."
6,24150919,# Assistez à une présentation d'un rapport de la Cour des comptes,"\n**Date** : Du 20 septembre 2025 à 08:30 au 21 septembre 2025 à 08:30\n**Lieu** : Palais Cambon - Cour des comptes,..."
7,40977521,# Journée Portes Ouvertes,\n**Date** : Le 17 janvier 2026 à 09:00\n**Lieu** : Institut Supérieur Clorivière- 119 Boulevard Diderot- 75012 PARI...
8,29502604,# Un patrimoine en constante évolution,"\n**Date** : Du 18 septembre 2026 à 09:00 au 19 septembre 2026 à 12:00\n**Lieu** : Gare du Nord, Paris\n**Adresse** ..."


In [16]:
explore_query("Quelle activité aquatique est proposée lors de l'événement 'On met les voiles !' ?")


REQUÊTE : « Quelle activité aquatique est proposée lors de l'événement 'On met les voiles !' ? »



,uid,titre,extrait
0,45384394,# On met les voiles !,"\n**Date** : Du 7 juin 2025 à 08:00 au 8 juin 2025 à 08:00\n**Lieu** : Domaine national du Palais-Royal, Paris\n**Ad..."
1,35775025,# Des mini-navires à piloter pour les enfants au bord de Seine avec MSC Croisières,"\n**Date** : Du 30 mai 2026 à 08:00 au 7 juin 2026 à 08:00\n**Lieu** : port de la tournelle, Paris\n**Adresse** : po..."
2,40948393,# Exposition des travaux des élèves des lycées professionnels et des écoles d’art de l’académie de Paris,"\n**Date** : Du 20 septembre 2025 à 08:00 au 21 septembre 2025 à 08:00\n**Lieu** : La Sorbonne, Paris\n**Adresse** :..."
3,9469400,# Soirée jazz manouche & swing / par Alba Obert,"\n**Date** : Le 6 juin 2026 à 17:30\n**Lieu** : JASS CLUB, Paris\n**Adresse** : 141 Rue de Tolbiac, Paris\n**Mots-cl..."
4,89411008,# Action collective,"\n**Date** : Le 28 mai 2026 à 09:00\n**Lieu** : Association Dessine-moi un mouton, Paris\n**Adresse** : 29 rue Pajol..."
5,84128307,"# Rencontre exceptionnelle avec Isabelle le Bot, Directrice générale de La France Mutualiste","\n**Date** : Le 6 novembre 2025 à 07:00\n**Lieu** : HEC Alumni, Paris\n**Adresse** : 9 avenue Franklin Roosevelt 750..."
6,5728980,# Choral Evensong for Fête de la Musique,"\n**Date** : Le 21 juin 2025 à 17:00\n**Lieu** : The American Cathedral in Paris, Paris\n**Adresse** : 23 avenue Geo..."
7,25627821,# Soeurs (Audrey & Victoria),"\n**Date** : Du 26 mai 2026 à 17:00 au 17 juin 2026 à 17:00\n**Lieu** : Théâtre de l'Atelier, Paris\n**Adresse** : 1..."
8,43402377,# Festival MVP 11 Après-midi-Echobat,"\n**Date** : Le 11 avril 2026 à 12:30\n**Lieu** : Paris - Agence Rosny-sous-Bois, Paris\n**Adresse** : 75019 Paris\n..."
9,80771862,# Festival MVP 11 après-midi - L'AFER,"\n**Date** : Le 11 avril 2026 à 12:30\n**Lieu** : Paris - Agence Longjumeau, Paris\n**Adresse** : 75019 Paris\n**Mot..."


In [17]:
explore_query("Quel est le niveau de diplôme requis pour postuler en tant que Conducteur de Trains SNCF ?")


REQUÊTE : « Quel est le niveau de diplôme requis pour postuler en tant que Conducteur de Trains SNCF ? »



,uid,titre,extrait
0,9548420,# Conducteur/Conductrice de Trains SNCF - Présentation du métier et du Processus de Recrutement,"\n**Date** : Le 9 février 2026 à 08:00\n**Lieu** : Paris - Agence Argenteuil, Paris\n**Adresse** : 75009 Paris\n**Mo..."
1,40977521,# Journée Portes Ouvertes,\n**Date** : Le 17 janvier 2026 à 09:00\n**Lieu** : Institut Supérieur Clorivière- 119 Boulevard Diderot- 75012 PARI...
2,47524672,# RECRUTEMENTS Agent(e)s de sécurité ferroviaire (H/F) - Gare du Nord - POEI 33 jours,"\n**Date** : Le 1 juillet 2026 à 07:30\n**Lieu** : Agence Paris 11eme Beaumarchais, Paris\n**Adresse** : 75011 Paris..."
3,53921440,# Concert d'un orchestre d'harmonie en Gare du Nord,"\n**Date** : Le 20 septembre 2025 à 09:00\n**Lieu** : Gare du Nord, Paris\n**Adresse** : 18 rue de Dunkerque 75010 P..."
4,68032223,# Concours scolaire Dis-moi dix mots,"\n**Date** : Le 21 mars 2026 à 17:00\n**Lieu** : Ministère de la Culture, Paris\n**Adresse** : 3 rue de Valois, 7500..."
5,77895516,# Présentation de la VAE pour le métier d'Assistant de vie et Agent service médico social,"\n**Date** : Le 21 novembre 2025 à 09:00\n**Lieu** : DT 75 Paris, Paris\n**Adresse** : 75012 Paris\n**Mots-clés** : ..."
6,99228039,# Journée Portes Ouvertes,"\n**Date** : Le 14 février 2026 à 09:00\n**Lieu** : EFHT - Ecole supérieure de Tourisme, Paris\n**Adresse** : 104 bo..."
7,29502604,# Un patrimoine en constante évolution,"\n**Date** : Du 18 septembre 2026 à 09:00 au 19 septembre 2026 à 12:00\n**Lieu** : Gare du Nord, Paris\n**Adresse** ..."
8,93649466,# Forum de recrutement de l'AP-HP,"\n**Date** : Le 12 février 2026 à 08:00\n**Lieu** : Cité Universitaire Internationale de Paris, Paris\n**Adresse** :..."
9,40948393,# Exposition des travaux des élèves des lycées professionnels et des écoles d’art de l’académie de Paris,"\n**Date** : Du 20 septembre 2025 à 08:00 au 21 septembre 2025 à 08:00\n**Lieu** : La Sorbonne, Paris\n**Adresse** :..."


In [18]:
explore_query("Faut-il payer ou s'inscrire pour participer aux Nuits des étoiles au parc Montsouris ?")


REQUÊTE : « Faut-il payer ou s'inscrire pour participer aux Nuits des étoiles au parc Montsouris ? »



,uid,titre,extrait
0,93263481,# Nuits des étoiles au parc Montsouris,"\n**Date** : Le 1 août 2025 à 19:00\n**Lieu** : Parc Montsouris, Paris\n**Adresse** : 2 rue Gazan 75014 Paris\n**Mot..."
1,80331120,# Soirée d’observation du ciel,"\n**Date** : Le 3 août 2025 à 19:00\n**Lieu** : Square Louise Michel, Paris\n**Adresse** : 6 Pl. Saint-Pierre, 75018..."
2,9469400,# Soirée jazz manouche & swing / par Alba Obert,"\n**Date** : Le 6 juin 2026 à 17:30\n**Lieu** : JASS CLUB, Paris\n**Adresse** : 141 Rue de Tolbiac, Paris\n**Mots-cl..."
3,21929520,# Musique au Jardin d'hiver,"\n**Date** : Le 21 juin 2026 à 11:00\n**Lieu** : Musée national Jean Jacques Henner, Paris\n**Adresse** : 43 Avenue ..."
4,45886037,# Hommage à Elvis Presley – Matthieu Boré Quartet,"\n**Date** : Le 21 juin 2026 à 19:00\n**Lieu** : Sunset Sunside, Paris\n**Adresse** : 60 rue des Lombards, 75001 Par..."
5,15139507,# Ouverture exceptionnelle de la Cathédrale Notre-Dame de Paris,"\n**Date** : Du 20 septembre 2025 à 06:15 au 21 septembre 2025 à 06:15\n**Lieu** : Cathédrale Notre-Dame de Paris, P..."
6,40977521,# Journée Portes Ouvertes,\n**Date** : Le 17 janvier 2026 à 09:00\n**Lieu** : Institut Supérieur Clorivière- 119 Boulevard Diderot- 75012 PARI...


In [19]:
explore_query("Quels sont les tarifs pour le spectacle 'Fiesta' au Théâtre de Verdure du jardin Shakespeare ?")


REQUÊTE : « Quels sont les tarifs pour le spectacle 'Fiesta' au Théâtre de Verdure du jardin Shakespeare ? »



,uid,titre,extrait
0,42804640,# Fiesta,\n**Date** : Du 5 juillet 2025 à 09:00 au 19 juillet 2025 à 09:00\n**Lieu** : Théâtre de Verdure du jardin Shakespea...
1,21929520,# Musique au Jardin d'hiver,"\n**Date** : Le 21 juin 2026 à 11:00\n**Lieu** : Musée national Jean Jacques Henner, Paris\n**Adresse** : 43 Avenue ..."
2,2102612,# Rendez-vous aux jardins au musée Bourdelle : Performance dansée « PURE » dans les jardins,"\n**Date** : Du 6 juin 2026 à 13:00 au 7 juin 2026 à 14:30\n**Lieu** : Musée Bourdelle, Paris\n**Adresse** : 18 Rue ..."
3,74636399,# Festival Porte Voix,"\n**Date** : Du 3 juillet 2026 à 18:00 au 4 juillet 2026 à 22:00\n**Lieu** : Point Ephémère, Paris\n**Adresse** : 20..."
4,45384394,# On met les voiles !,"\n**Date** : Du 7 juin 2025 à 08:00 au 8 juin 2025 à 08:00\n**Lieu** : Domaine national du Palais-Royal, Paris\n**Ad..."
5,51382688,# Hercule(s) (7-12 ans en famille),"\n**Date** : Du 8 août 2025 à 12:30 au 13 août 2025 à 12:30\n**Lieu** : Musée des arts et métiers, Paris\n**Adresse*..."
6,25627821,# Soeurs (Audrey & Victoria),"\n**Date** : Du 26 mai 2026 à 17:00 au 17 juin 2026 à 17:00\n**Lieu** : Théâtre de l'Atelier, Paris\n**Adresse** : 1..."
7,10335842,"# Ateliers Créatifs- ""La Fête au Jardin"" et ""Custom post""","\n**Date** : Le 28 juin 2026 à 13:00\n**Lieu** : Jardin 21, Paris\n**Adresse** : 12 rue Ella Fitzgerald 75019 Paris\..."


In [20]:
explore_query("Quel équipement est fourni aux auditeurs lors de l'expérience sonore 'Les Sons de Voyager' ?")


REQUÊTE : « Quel équipement est fourni aux auditeurs lors de l'expérience sonore 'Les Sons de Voyager' ? »



,uid,titre,extrait
0,8447931,# Les Sons de Voyager,"\n**Date** : Le 4 octobre 2025 à 08:30\n**Lieu** : Médiathèque Marguerite Duras, Paris\n**Adresse** : 115 Rue de Bag..."
1,28560513,"# Atelier ""Sons et parfums d'ailleurs""","\n**Date** : Le 26 avril 2026 à 14:30\n**Lieu** : Holivita, Paris\n**Adresse** : Holivita\n**Mots-clés** : ['Nature'..."
2,56318505,# Atelier Bulles en musique,"\n**Date** : Le 28 juin 2025 à 12:00\n**Lieu** : Place Marthe Simard, Paris\n**Adresse** : Place Marthe Simard, 7501..."
3,83539056,# Visite guidée des salons d'apparat et de l'Atelier de l'hôtel de la Fondation Singer-Polignac,"\n**Date** : Le 19 septembre 2026 à 08:30\n**Lieu** : Fondation Singer-Polignac, Paris\n**Adresse** : 43 avenue Geor..."
4,74636399,# Festival Porte Voix,"\n**Date** : Du 3 juillet 2026 à 18:00 au 4 juillet 2026 à 22:00\n**Lieu** : Point Ephémère, Paris\n**Adresse** : 20..."
5,7087528,"# Concert de Vox Cordis - Guitare, chant - ODYSSÉE","\n**Date** : Le 21 septembre 2025 à 15:00\n**Lieu** : Chapelle des sœurs auxiliatrices, Paris\n**Adresse** : 16 rue ..."
6,24150919,# Assistez à une présentation d'un rapport de la Cour des comptes,"\n**Date** : Du 20 septembre 2025 à 08:30 au 21 septembre 2025 à 08:30\n**Lieu** : Palais Cambon - Cour des comptes,..."
7,25627821,# Soeurs (Audrey & Victoria),"\n**Date** : Du 26 mai 2026 à 17:00 au 17 juin 2026 à 17:00\n**Lieu** : Théâtre de l'Atelier, Paris\n**Adresse** : 1..."


In [21]:
explore_query("Qui animera la scène TWITCH lors du Festival MVP - Ma Vie Pro ?")


REQUÊTE : « Qui animera la scène TWITCH lors du Festival MVP - Ma Vie Pro ? »



,uid,titre,extrait
0,80771862,# Festival MVP 11 après-midi - L'AFER,"\n**Date** : Le 11 avril 2026 à 12:30\n**Lieu** : Paris - Agence Longjumeau, Paris\n**Adresse** : 75019 Paris\n**Mot..."
1,92899699,# FESTIVAL MVP 11 MATIN - COMPLEA BIENVIEILLIR,"\n**Date** : Le 11 avril 2026 à 08:00\n**Lieu** : Paris - Agence Saint-Maur-des-Fosses, Paris\n**Adresse** : 75019 P..."
2,92590257,# Festival MVP 11 matin - MONOPRIX,"\n**Date** : Le 11 avril 2026 à 09:00\n**Lieu** : Paris - Agence Clichy-la-Garenne, Paris\n**Adresse** : 75019 Paris..."
3,88939268,# Festival MVP 11 Matin - ICTS,"\n**Date** : Le 11 avril 2026 à 08:00\n**Lieu** : Paris - DT 95 Val-d'Oise, Paris\n**Adresse** : 75019 Paris\n**Mots..."
4,43402377,# Festival MVP 11 Après-midi-Echobat,"\n**Date** : Le 11 avril 2026 à 12:30\n**Lieu** : Paris - Agence Rosny-sous-Bois, Paris\n**Adresse** : 75019 Paris\n..."
5,24291431,# Festival MVP 11 Matin - SERIS SERVICES AIRPORT,"\n**Date** : Le 11 avril 2026 à 08:00\n**Lieu** : Paris - DT 95 Val-d'Oise, Paris\n**Adresse** : 75019 Paris\n**Mots..."
6,74636399,# Festival Porte Voix,"\n**Date** : Du 3 juillet 2026 à 18:00 au 4 juillet 2026 à 22:00\n**Lieu** : Point Ephémère, Paris\n**Adresse** : 20..."
7,9469400,# Soirée jazz manouche & swing / par Alba Obert,"\n**Date** : Le 6 juin 2026 à 17:30\n**Lieu** : JASS CLUB, Paris\n**Adresse** : 141 Rue de Tolbiac, Paris\n**Mots-cl..."
8,41290551,# La Panza fête la musique 2025,"\n**Date** : Le 21 juin 2025 à 16:00\n**Lieu** : LA PANZA, Paris\n**Adresse** : 43 rue servan 75011 Paris\n\nLes art..."


In [22]:
explore_query("Pourquoi l'accès aux tribunes est-il interdit à l'Église Saint-Vincent-de-Paul ?")


REQUÊTE : « Pourquoi l'accès aux tribunes est-il interdit à l'Église Saint-Vincent-de-Paul ? »



,uid,titre,extrait
0,15139507,# Ouverture exceptionnelle de la Cathédrale Notre-Dame de Paris,"\n**Date** : Du 20 septembre 2025 à 06:15 au 21 septembre 2025 à 06:15\n**Lieu** : Cathédrale Notre-Dame de Paris, P..."
1,53552186,# Un joyau du 19e siècle,"\n**Date** : Le 21 septembre 2025 à 12:00\n**Lieu** : Église Saint-Vincent-de-Paul, Paris\n**Adresse** : Place Franz..."
2,85254602,# Visite guidée « Histoire et personnalités LGBT du cimetière du Père Lachaise »,"\n**Date** : Le 21 septembre 2025 à 08:30\n**Lieu** : Cimetière du Père Lachaise, Paris\n**Adresse** : 8 boulevard d..."
3,79293186,# Portes ouvertes à Bon Secours,"\n**Date** : Le 20 septembre 2025 à 08:00\n**Lieu** : Église luthérienne de Bon-Secours, Paris\n**Adresse** : 20 Rue..."
4,24150919,# Assistez à une présentation d'un rapport de la Cour des comptes,"\n**Date** : Du 20 septembre 2025 à 08:30 au 21 septembre 2025 à 08:30\n**Lieu** : Palais Cambon - Cour des comptes,..."
5,65315210,# Visite du site historique du Collège de France,"\n**Date** : Du 20 septembre 2025 à 08:00 au 21 septembre 2025 à 08:00\n**Lieu** : Collège de France, Paris\n**Adres..."
6,83539056,# Visite guidée des salons d'apparat et de l'Atelier de l'hôtel de la Fondation Singer-Polignac,"\n**Date** : Le 19 septembre 2026 à 08:30\n**Lieu** : Fondation Singer-Polignac, Paris\n**Adresse** : 43 avenue Geor..."
7,40977521,# Journée Portes Ouvertes,\n**Date** : Le 17 janvier 2026 à 09:00\n**Lieu** : Institut Supérieur Clorivière- 119 Boulevard Diderot- 75012 PARI...


In [23]:
explore_query("Quelles sont les quatre figures irlandaises de l'exposition 'Amitiés littéraires' au Centre Culturel Irlandais ?")


REQUÊTE : « Quelles sont les quatre figures irlandaises de l'exposition 'Amitiés littéraires' au Centre Culturel Irlandais ? »



,uid,titre,extrait
0,68942455,"# Exposition ""Amitiés littéraires : florilège de la Bibliothèque littéraire Jacques Doucet""","\n**Date** : Du 20 septembre 2025 à 12:00 au 21 septembre 2025 à 12:00\n**Lieu** : Centre Culturel Irlandais, Paris\..."
1,83035840,# Visite de l'Ambassade d'Irlande en France,"\n**Date** : Le 20 septembre 2025 à 08:00\n**Lieu** : Ambassade d'Irlande, Paris\n**Adresse** : 2 rue Rude 75116 Par..."
2,82056401,# Soirée littéraire autour de Jana Černá,"\n**Date** : Le 12 février 2026 à 18:00\n**Lieu** : Centre tchèque de Paris, Paris\n**Adresse** : 18 Rue Bonaparte, ..."
3,14393282,"# Poésie d'Afrique au Sud du Sahara, avec Bernard Magnier et Alain Mabanckou","\n**Date** : Le 25 mars 2026 à 18:00\n**Lieu** : Alliance française de Paris, Paris\n**Adresse** : 101 boulevard Ras..."
4,35646481,# Vernissage de l'exposition de Livres d'artiste France-Brésil : M.A.L.A. (Moradia Andarilha de Livros de Artista),"\n**Date** : Le 3 octobre 2025 à 16:30\n**Lieu** : Médiathèque Marguerite Duras, Paris\n**Adresse** : 115 Rue de Bag..."
5,63549255,# Exposition - Galerie Clémentine de la Féronnière,"\n**Date** : Du 1 octobre 2026 à 09:00 au 14 novembre 2026 à 10:00\n**Lieu** : Galerie Clémentine de la Féronnière, ..."
6,53552186,# Un joyau du 19e siècle,"\n**Date** : Le 21 septembre 2025 à 12:00\n**Lieu** : Église Saint-Vincent-de-Paul, Paris\n**Adresse** : Place Franz..."
7,40948393,# Exposition des travaux des élèves des lycées professionnels et des écoles d’art de l’académie de Paris,"\n**Date** : Du 20 septembre 2025 à 08:00 au 21 septembre 2025 à 08:00\n**Lieu** : La Sorbonne, Paris\n**Adresse** :..."


## Détail d'un parent document

Pour inspecter le contenu complet d'un événement, renseigner son `uid` ci-dessous.

In [24]:
uids = [
    # Renseigner les UIDs à inspecter
]

for uid in uids:
    doc = parent_docs.get(uid)
    if doc:
        print(doc)
    else:
        print(f"UID {uid} non trouvé.")
    print()